# Regressão linear — o caso da limonada

**Capítulo II.2** do livro vivo [Ciência de Dados e Aprendizado de Máquina](https://machinelearning.ghdaru.com.br/ii-2-modelos-lineares.html).

365 dias de uma barraca de limonada. Você vai:

1. tirar a conclusão errada, como quase todo relatório tira;
2. descobrir **por que** ela é errada, olhando o dado;
3. tentar o conserto de manual — e ver que ele **não conserta**;
4. chegar à resposta honesta, que é a menos confortável.

Roda **sem instalar nada**: só a biblioteca padrão do Python e o código do próprio livro.

In [ ]:
# --- roda igual na sua máquina e no Colab ------------------------------
# Na sua máquina: o notebook acha o repositório subindo de pasta.
# No Colab: não há repositório, então os arquivos necessários são baixados.
import pathlib, sys, urllib.request

RAW = "https://raw.githubusercontent.com/GHDaru/machinelearning/main/"
PRECISA = ['ml-zero/etapa-05/linear.py', 'ml-zero/dados/limonada/limonada.csv']

raiz = pathlib.Path.cwd()
for _ in range(5):
    if (raiz / "ml-zero").is_dir():
        break
    raiz = raiz.parent
else:
    raiz = pathlib.Path.cwd()

for rel in PRECISA:
    destino = raiz / rel
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(RAW + rel, destino)
        print("baixado:", rel)

sys.path.insert(0, str(raiz / "ml-zero/etapa-05"))
RAIZ = raiz
print("pronto.")

## 1. O dado

Sete colunas, 365 linhas, nenhum valor faltante.

In [ ]:
import csv

with open(RAIZ / "ml-zero/dados/limonada/limonada.csv", encoding="utf8") as f:
    linhas = list(csv.DictReader(f))

ATRIBUTOS = ["temperatura", "precipitacao", "panfletos", "preco"]
X = [[float(l[a]) for a in ATRIBUTOS] for l in linhas]
y = [float(l["vendas"]) for l in linhas]

print(len(linhas), "dias ·", linhas[0]["data"], "a", linhas[-1]["data"])
print(linhas[0])

## 2. A leitura ingênua

A primeira coisa que se faz: correlacionar cada atributo com a resposta.

In [ ]:
def correlacao(a, b):
    ma, mb = sum(a) / len(a), sum(b) / len(b)
    num = sum((x - ma) * (z - mb) for x, z in zip(a, b))
    den = (sum((x - ma) ** 2 for x in a) * sum((z - mb) ** 2 for z in b)) ** 0.5
    return num / den if den else float("nan")

for i, nome in enumerate(ATRIBUTOS):
    coluna = [linha[i] for linha in X]
    print(f"{nome:14s} r = {correlacao(coluna, y):+.3f}")

Calor vende, chuva atrapalha, panfleto ajuda.

E **preço mais alto vende mais** — `r = +0,513`.

Essa última linha é uma recomendação de negócio esperando para ser escrita num slide: *aumente o preço*. Antes de escrevê-la, olhe o dado.

In [ ]:
from collections import defaultdict

por_preco = defaultdict(list)
for linha, l in zip(X, linhas):
    por_preco[l["preco"]].append((linha[0], float(l["vendas"]), int(l["data"][5:7])))

for preco in sorted(por_preco):
    dias = por_preco[preco]
    temp = sum(d[0] for d in dias) / len(dias)
    vend = sum(d[1] for d in dias) / len(dias)
    meses = sorted({d[2] for d in dias})
    print(f"preço {preco} · {len(dias):3d} dias · temp média {temp:5.1f} · "
          f"vendas médias {vend:5.1f} · meses {meses}")

## 3. O que estava acontecendo

O preço de 0,50 aparece **só em julho e agosto**. O preço subiu no verão.

`preco` não é uma alavanca de decisão: é um **termômetro disfarçado**. A correlação de +0,513 mede o calor de julho, não a disposição do freguês a pagar.

## 4. O conserto de manual

"Controle pelas outras variáveis." Ajustando tudo junto, com a implementação do próprio livro:

In [ ]:
from linear import RegressaoLinear

# padronizar=False para os coeficientes saírem na unidade original —
# é o que permite lê-los como "copo por grau", "copo por panfleto".
modelo = RegressaoLinear(solucao_fechada=True, padronizar=False).fit(X, y)

print(f"intercepto    {modelo.vies:+8.4f}")
for nome, w in zip(ATRIBUTOS, modelo.pesos):
    print(f"{nome:14s}{w:+8.4f}")

pred = modelo.predict(X)
my = sum(y) / len(y)
r2 = 1 - sum((a - b) ** 2 for a, b in zip(y, pred)) / sum((v - my) ** 2 for v in y)
print(f"\nR² = {r2:.4f}")

**O coeficiente do preço continua positivo.**

Controlar pela temperatura não desfez nada — porque a temperatura média do dia não captura *ser julho* (férias, fluxo de rua, hábito), e o que sobrou disso continua morando dentro de `preco`.

> **Controlar por uma variável só remove o confundimento que aquela variável mede.**

E repare no R²: 0,982. Nenhuma métrica avisou.

## 5. Inverter o coeficiente devolve a unidade da decisão

`0,0188 copo por panfleto` não diz nada a quem manda imprimir panfleto.

In [ ]:
coef_panfleto = modelo.pesos[ATRIBUTOS.index("panfletos")]
print(f"{1 / coef_panfleto:.0f} panfletos para vender um copo a mais")
print(f"correlação temperatura × panfletos: {correlacao([l[0] for l in X], [l[2] for l in X]):+.3f}")

53 panfletos por copo — e a panfletagem provavelmente não se paga.

Só que `panfletos` correlaciona +0,798 com `temperatura`: em dia quente distribuíam-se mais panfletos. Parte desses 0,0188 é calor, não panfleto. O efeito real da panfletagem é **ainda menor**.

## 6. Agora tente o conserto óbvio

Isolar um período em que o preço varie **sem a estação variar junto**, e ajustar só ali.

In [ ]:
precos_por_mes = defaultdict(set)
for l in linhas:
    precos_por_mes[int(l["data"][5:7])].add(l["preco"])

for mes in sorted(precos_por_mes):
    print(f"mês {mes:2d}: preços distintos = {len(precos_por_mes[mes])}  {sorted(precos_por_mes[mes])}")

print("\nmeses com mais de um preço:",
      [m for m in precos_por_mes if len(precos_por_mes[m]) > 1] or "NENHUM")

**Nenhum mês tem mais de um preço.**

Restringir a julho e agosto não isola o efeito do preço: deixa o preço **constante** — e atributo que não varia não tem coeficiente.

O confundimento aqui é **perfeito**: preço e estação são a mesma variável com dois nomes. Não há recorte, controle nem modelo que as separe. **A informação não está no dado.**

## 7. E o R², enfim, avaliado com honestidade

Até aqui o R² foi medido nos mesmos dados do ajuste. Como isto é uma série diária, a divisão tem de respeitar o tempo — treinar no passado, testar no futuro.

In [ ]:
corte = 300   # treina nos 300 primeiros dias, testa nos 65 últimos
m2 = RegressaoLinear(solucao_fechada=True, padronizar=False).fit(X[:corte], y[:corte])

def r2_de(modelo, X_, y_):
    p = modelo.predict(X_)
    m = sum(y_) / len(y_)
    return 1 - sum((a - b) ** 2 for a, b in zip(y_, p)) / sum((v - m) ** 2 for v in y_)

print(f"R² no treino (dias 1–{corte})      {r2_de(m2, X[:corte], y[:corte]):.4f}")
print(f"R² no teste  (dias {corte + 1}–365)    {r2_de(m2, X[corte:], y[corte:]):.4f}")

## O que levar

- A correlação do preço era **real** e a conclusão era **falsa**. As duas coisas ao mesmo tempo.
- Controlar por variáveis remove só o confundimento que elas medem.
- O R² alto não protege de nada disso — ele nem piscou.
- A resposta honesta pode ser *"com estes dados não dá"*, acompanhada de **o que precisaria ser coletado**: preço variando dentro do mesmo mês.

Continue no capítulo: [05 — Modelos Lineares](https://machinelearning.ghdaru.com.br/ii-2-modelos-lineares.html#o-caso-da-limonada), e responda os exercícios `modelos-lineares-e4`, `modelos-lineares-e5` e `modelos-lineares-e6` — eles são corrigidos no servidor, com a explicação completa na segunda tentativa.